# Ensemble of ensembles

## Classification

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc
from lightgbm import LGBMClassifier

task = "Classification"
dataset_size = "100k"

# Data loading
test_data_path = Path("data") / "test_data.parquet"
preds_path = Path("test_results") / task / "predictions" / f"test_preds_{dataset_size}_tuned.parquet"
X_test_base = pd.read_parquet(test_data_path)
df_preds = pd.read_parquet(preds_path)

# Data preprocessing
targets = ["target", "target_annual_roi"]
X_test_base = X_test_base.drop(columns=targets)
explicit_drop = ["issue_d", "earliest_cr_line"]
X_test_base = X_test_base.drop(columns=explicit_drop)

# Use predictions of models excluding Dummy
all_models = df_preds.columns
valid_models = all_models[~all_models.str.contains("Dummy")]
valid_models = valid_models.drop("true_y", errors='ignore')
y_true = df_preds["true_y"]

# Combine base features with valid model predictions
base_features = X_test_base.reset_index(drop=True)
model_predictions = df_preds[valid_models].reset_index(drop=True)
df_full = pd.concat([base_features, model_predictions], axis=1)

# Chronological split: 80% train, 20% test
X_train_full, X_test_full, y_train, y_test = train_test_split(
    df_full, y_true, test_size=0.20, shuffle=False
)

# Drop model prediction columns for raw data experiment
X_train_raw = X_train_full.drop(columns=valid_models)
X_test_raw = X_test_full.drop(columns=valid_models)

# LightGBM parameters
lgbm_params = {
    "boosting_type": "goss",
    "num_leaves": 444,
    "max_depth": 15,
    "learning_rate": 0.0013490712742580085,
    "scale_pos_weight": 2.529589934768151,
    "min_split_gain": 3.4510864669147536,
    "min_child_weight": 0.19239271728859866,
    "min_child_samples": 234,
    "colsample_bytree": 0.31324120666189326,
    "reg_alpha": 0.6545836243558268,
    "reg_lambda": 0.1069421692665621,
    "colsample_bynode": 0.21348019642019067,
    "min_data_per_group": 803,
    "max_cat_threshold": 819,
    "cat_l2": 0.00011204200863439848,
    "cat_smooth": 1.5621591746591863,
    "max_cat_to_onehot": 36,
    "max_bin": 258,
    "n_estimators": 8720,
    "top_rate": 0.24249410683009498,
    "other_rate": 0.42369427528780756,
    "random_state": 42,
    "n_jobs": -1,
    "verbose": -1,
    "objective": "binary",
    "metric": "auc",
}

# Evaluation of base models
print("Original Model Performance on Test Set:")
for model in valid_models:
    roc = roc_auc_score(y_test, X_test_full[model]) 
    precision, recall, _ = precision_recall_curve(y_test, X_test_full[model])
    pr_auc = auc(recall, precision)
    print(f"- {model:<12} ROC AUC: {roc:.4f} | PR AUC: {pr_auc:.4f}")

# 1. Evaluate LightGBM on original features
print("\n Original features only (newly trained LightGBM):")
model_raw = LGBMClassifier(**lgbm_params)
model_raw.fit(X_train_raw, y_train)
preds_raw = model_raw.predict_proba(X_test_raw)[:, 1]

roc_raw = roc_auc_score(y_test, preds_raw)
precision, recall, _ = precision_recall_curve(y_test, preds_raw)
pr_raw = auc(recall, precision)

print(f"ROC AUC:\t{roc_raw:.4f}")
print(f"PR AUC:\t\t{pr_raw:.4f}")

# Relative Feature Importances for Model 1
print("\nTop 15 Feature Importances (Original features):")
raw_importances = model_raw.feature_importances_
feat_imp_raw = pd.DataFrame({
    'Feature': X_train_raw.columns, 
    'Importance (%)': (raw_importances / raw_importances.sum()) * 100
}).sort_values(by='Importance (%)', ascending=False).head(15)
display(feat_imp_raw.style.hide(axis="index").format({'Importance (%)': '{:.2f}%'}))


# 2. Evaluate LightGBM on original features + predictions
print("\n Original features + model predictions (newly trained LightGBM):")
model_meta = LGBMClassifier(**lgbm_params)
model_meta.fit(X_train_full, y_train)
preds_meta = model_meta.predict_proba(X_test_full)[:, 1]

roc_meta = roc_auc_score(y_test, preds_meta)
precision, recall, _ = precision_recall_curve(y_test, preds_meta)
pr_meta = auc(recall, precision)

print(f"ROC AUC:\t{roc_meta:.4f}")
print(f"PR AUC:\t\t{pr_meta:.4f}")

# Relative Feature Importances for Model 2
print("\nTop 15 Feature Importances (Original features + Predictions):")
meta_importances = model_meta.feature_importances_
feat_imp_meta = pd.DataFrame({
    'Feature': X_train_full.columns, 
    'Importance (%)': (meta_importances / meta_importances.sum()) * 100
}).sort_values(by='Importance (%)', ascending=False).head(15)
display(feat_imp_meta.style.hide(axis="index").format({'Importance (%)': '{:.2f}%'}))

Original Model Performance on Test Set:
- XGBoost      ROC AUC: 0.7324 | PR AUC: 0.3175
- LightGBM     ROC AUC: 0.7335 | PR AUC: 0.3199
- CatBoost     ROC AUC: 0.7315 | PR AUC: 0.3156
- NGBoost      ROC AUC: 0.7323 | PR AUC: 0.3199
- GBM          ROC AUC: 0.7232 | PR AUC: 0.3075
- HistGBM      ROC AUC: 0.7316 | PR AUC: 0.3181
- TabPFN       ROC AUC: 0.7316 | PR AUC: 0.3183

 Original features only (newly trained LightGBM):
ROC AUC:	0.7505
PR AUC:		0.3438

Top 15 Feature Importances (Original features):


Feature,Importance (%)
dti,3.11%
installment,2.98%
annual_inc,2.66%
loan_amnt,2.66%
bc_util,2.65%
addr_state,2.53%
mo_sin_old_il_acct,2.51%
bc_open_to_buy,2.45%
mo_sin_old_rev_tl_op,2.45%
il_util,2.39%



 Original features + model predictions (newly trained LightGBM):
ROC AUC:	0.7510
PR AUC:		0.3468

Top 15 Feature Importances (Original features + Predictions):


Feature,Importance (%)
TabPFN,2.94%
addr_state,2.82%
mo_sin_old_il_acct,2.70%
annual_inc,2.57%
installment,2.57%
bc_util,2.51%
bc_open_to_buy,2.41%
dti,2.38%
il_util,2.21%
CatBoost,2.20%


## Regression

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor

task = "Regression"
dataset_size = "100k"

# Data loading
test_data_path = Path("data") / "test_data.parquet"
preds_path = Path("test_results") / task / "predictions" / f"test_preds_{dataset_size}_tuned.parquet"
X_test_base = pd.read_parquet(test_data_path)
df_preds = pd.read_parquet(preds_path)

# Data preprocessing
targets = ["target", "target_annual_roi", "loan_status"]
X_test_base = X_test_base.drop(columns=[c for c in targets if c in X_test_base.columns])

datetime_cols = X_test_base.select_dtypes(include=['datetime', 'datetimetz']).columns
if len(datetime_cols) > 0:
    X_test_base = X_test_base.drop(columns=datetime_cols)

explicit_drop = ["issue_d", "earliest_cr_line"]
X_test_base = X_test_base.drop(columns=[c for c in explicit_drop if c in X_test_base.columns])

# Convert object columns to category format for XGBoost (enable_categorical=True)
object_cols = X_test_base.select_dtypes(include=['object', 'string']).columns
for col in object_cols:
    X_test_base[col] = X_test_base[col].astype('category')

# Use predictions of models excluding Dummy and specific ensemble metrics
exclude_cols = ["true_y", "Ensemble_Mean", "Ensemble"] 
valid_models = pd.Index([col for col in df_preds.columns if col not in exclude_cols and "Dummy" not in col])
y_true = df_preds["true_y"]

# Combine base features with valid model predictions
base_features = X_test_base.reset_index(drop=True)
model_predictions = df_preds[valid_models].reset_index(drop=True)
df_full = pd.concat([base_features, model_predictions], axis=1)

# Chronological split: 80% train, 20% test
X_train_full, X_test_full, y_train, y_test = train_test_split(
    df_full, y_true, test_size=0.20, shuffle=False
)

# Drop model prediction columns for raw data experiment
X_train_raw = X_train_full.drop(columns=valid_models)
X_test_raw = X_test_full.drop(columns=valid_models)

# XGBoost parameters
xgboost_100k_reg_params = {
    "booster": "gbtree",
    "learning_rate": 0.00358874054592766,
    "min_split_loss": 0.30665873968801743,
    "max_depth": 9,
    "min_child_weight": 0.058208746224353215,
    "max_delta_step": 6.600824779885242,
    "colsample_bytree": 0.31452203665302875,
    "colsample_bylevel": 0.7845215630136946,
    "colsample_bynode": 0.45580656447386975,
    "reg_lambda": 1.4591079946865122,
    "reg_alpha": 7.300918108293281,
    "grow_policy": "depthwise",
    "max_leaves": 311,
    "max_bin": 270,
    "max_cat_to_onehot": 25,
    "max_cat_threshold": 578,
    "n_estimators": 9240,
    "sampling_method": "uniform",
    "subsample": 0.892493539712304,
    "random_state": 42,
    "n_jobs": -1,
    "verbosity": 0,
    "tree_method": "hist",
    "objective": "reg:squarederror",
    "enable_categorical": True,
}

# Evaluation of base models
print("Original Model Performance on Test Set:")
for model in valid_models:
    rmse = np.sqrt(mean_squared_error(y_test, X_test_full[model]))
    mae = mean_absolute_error(y_test, X_test_full[model])
    r2 = r2_score(y_test, X_test_full[model])
    print(f"- {model:<12} RMSE: {rmse:.4f} | MAE: {mae:.4f} | R2: {r2:.4f}")

# 1. Evaluate XGBoost on original features
print("\n Original features only (newly trained XGBoost):")
model_raw = XGBRegressor(**xgboost_100k_reg_params)
model_raw.fit(X_train_raw, y_train)
preds_raw = model_raw.predict(X_test_raw)

rmse_raw = np.sqrt(mean_squared_error(y_test, preds_raw))
mae_raw = mean_absolute_error(y_test, preds_raw)
r2_raw = r2_score(y_test, preds_raw)

print(f"RMSE:\t{rmse_raw:.4f}")
print(f"MAE:\t{mae_raw:.4f}")
print(f"R2:\t{r2_raw:.4f}")

# Relative Feature Importances for Model 1
print("\nTop 15 Feature Importances (Original features):")
raw_importances = model_raw.feature_importances_
feat_imp_raw = pd.DataFrame({
    'Feature': X_train_raw.columns, 
    'Importance (%)': (raw_importances / raw_importances.sum()) * 100
}).sort_values(by='Importance (%)', ascending=False).head(15)
display(feat_imp_raw.style.hide(axis="index").format({'Importance (%)': '{:.2f}%'}))


# 2. Evaluate XGBoost on original features + predictions
print("\n Original features + model predictions (newly trained XGBoost):")
model_meta = XGBRegressor(**xgboost_100k_reg_params)
model_meta.fit(X_train_full, y_train)
preds_meta = model_meta.predict(X_test_full)

rmse_meta = np.sqrt(mean_squared_error(y_test, preds_meta))
mae_meta = mean_absolute_error(y_test, preds_meta)
r2_meta = r2_score(y_test, preds_meta)

print(f"RMSE:\t{rmse_meta:.4f}")
print(f"MAE:\t{mae_meta:.4f}")
print(f"R2:\t{r2_meta:.4f}")

# Relative Feature Importances for Model 2
print("\nTop 15 Feature Importances (Original features + Predictions):")
meta_importances = model_meta.feature_importances_
feat_imp_meta = pd.DataFrame({
    'Feature': X_train_full.columns, 
    'Importance (%)': (meta_importances / meta_importances.sum()) * 100
}).sort_values(by='Importance (%)', ascending=False).head(15)
display(feat_imp_meta.style.hide(axis="index").format({'Importance (%)': '{:.2f}%'}))

Original Model Performance on Test Set:
- XGBoost      RMSE: 0.3808 | MAE: 0.2550 | R2: 0.0578
- LightGBM     RMSE: 0.3810 | MAE: 0.2554 | R2: 0.0568
- CatBoost     RMSE: 0.3809 | MAE: 0.2561 | R2: 0.0573
- NGBoost      RMSE: 0.3825 | MAE: 0.2572 | R2: 0.0495
- GBM          RMSE: 0.3822 | MAE: 0.2566 | R2: 0.0511
- HistGBM      RMSE: 0.3822 | MAE: 0.2570 | R2: 0.0510
- PGBM         RMSE: 0.3820 | MAE: 0.2568 | R2: 0.0520
- TabPFN       RMSE: 0.3826 | MAE: 0.2505 | R2: 0.0490

 Original features only (newly trained XGBoost):
RMSE:	0.3774
MAE:	0.2644
R2:	0.0748

Top 15 Feature Importances (Original features):


Feature,Importance (%)
num_tl_120dpd_2m,7.21%
term_months,5.79%
home_ownership,2.76%
mort_acc,2.20%
int_rate,2.02%
sub_grade,1.92%
emp_length,1.83%
addr_state,1.70%
application_type,1.60%
dti,1.58%



 Original features + model predictions (newly trained XGBoost):
RMSE:	0.3776
MAE:	0.2641
R2:	0.0737

Top 15 Feature Importances (Original features + Predictions):


Feature,Importance (%)
num_tl_120dpd_2m,3.75%
XGBoost,3.22%
LightGBM,3.20%
CatBoost,3.00%
GBM,2.67%
term_months,2.49%
HistGBM,2.48%
TabPFN,2.29%
PGBM,2.08%
NGBoost,2.06%
